# 05 — Evidence-grounded LLM report generation

**Objective.** Turn one validated feature record, plus the compatible classifier output and retrieved
reference passages, into a structured JSON report. The design is **"code states the facts, the LLM
explains them"**:

| report field | filled by |
|---|---|
| `observed_features`, `quantitative_radiomics_summary`, `classifier_output`, `birads_assessment`, `requires_radiologist_review` | deterministic code (from the record + classifier) |
| `summary`, `supporting_evidence`, `uncertainty_and_limitations`, `recommended_review` | LLM, then validated |

The validator (`bus_research.report.validate_narrative`) rejects narratives that cite unretrieved
chunks, state a BI-RADS category, issue biopsy/surgery directives, quote numbers not in the facts, or
describe unobserved lexicon findings. Rejected drafts are retried, then reported as `invalid`, never
shown. If extraction or mask QC failed, the system **abstains** without calling the LLM.

RAG supplies *evidence*. It does not train or fine-tune the LLM.

**Prerequisites.** Notebooks 02–04. **Outputs.** `results/reports/research/*.json|.md`.

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

# Resolve the repo root whether the kernel starts in notebooks/ or at the root:
# config.yaml paths (dataset/, checkpoints/, results/) are relative to it.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "config.yaml").exists() and (p / "bus_research").is_dir())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.config import load_config
from utils.seed import get_device, set_seed

config = load_config(str(REPO_ROOT / "config.yaml"))
set_seed(config["seed"], config["deterministic"])
RESEARCH = config["research"]
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/mulatyazewchekol/Documents/My_Projects/Healthcare_Projects/Breast_Cancer_Detection


In [2]:
import bus_research

missing = bus_research.missing_dependencies()
print("Missing research dependencies:", missing or "none")
# If anything is missing, install the pinned optional set (see the comments in that file):
# %pip install -r requirements-research.txt

Missing research dependencies: none


In [3]:
import copy
import time

from bus_research import rag
from bus_research import report as rp
from bus_research.classifier import load_if_present
from bus_research.feature_store import atomic_write_json, load_record

rad_dir = REPO_ROOT / RESEARCH["radiomics"]["output_dir"]
manifest = json.loads((rad_dir / "manifest.json").read_text())
index = rag.ReferenceIndex.load(REPO_ROOT / RESEARCH["rag"]["index_dir"])
clf = load_if_present(REPO_ROOT / RESEARCH["models"]["classifier_dir"])
out_dir = REPO_ROOT / RESEARCH["reports"]["output_dir"]
out_dir.mkdir(parents=True, exist_ok=True)
print("RAG index:", index.version, "| classifier:", clf.model_id if clf else "not available")

RAG index: rag-c47fd3b06bdb | classifier: radiomics-gradient-boosting-c622ae6f


## 1. Inference view of a record

Reports are generated from **real-inference** (`cascade`) test records. The dataset label inside the
stored record is removed before anything else touches it. That is exactly what the app has at
inference time: no label.

In [4]:
def inference_view(record):
    rec = copy.deepcopy(record)
    rec["reference_labels"] = {"benign_malignant": None, "birads_category": None, "label_source": None, "split": None}
    return rec

test_cascade = [e for e in manifest["records"] if e["variant"] == "cascade" and e["split"] == "test"]
records = [inference_view(load_record(rad_dir / e["record_path"])) for e in test_cascade[:2]]
for rec in records:
    pred = clf.predict(rec) if clf else {"available": False, "reason": "no trained artifact"}
    rec["_prediction"] = pred
    print(rec["record_id"], "| QC:", rec["segmentation"]["quality_status"], "| p(malignant):",
          round(pred["class_probabilities"]["malignant"], 3) if pred.get("available") else pred.get("reason"),
          "| calibration:", pred.get("calibration_status"), "| OOD:", (pred.get("ood") or {}).get("out_of_distribution"))

rec_859f5f43dfe2 | QC: pass | p(malignant): 0.396 | calibration: calibrated | OOD: False
rec_4eb14bcaed4f | QC: pass | p(malignant): 0.19 | calibration: calibrated | OOD: False


## 2. Retrieval for this record

In [5]:
def retrieve(rec, pred):
    query = rag.build_query(rec, pred)
    hits = index.search(query, k=RESEARCH["rag"]["top_k"])
    primary = index.search(query, k=2, tiers=("primary_reference",))   # always offer the ACR card
    return list({h["chunk_id"]: h for h in primary + hits}.values())

hits = retrieve(records[0], records[0]["_prediction"])
print("query:", rag.build_query(records[0], records[0]["_prediction"]))
for h in hits:
    print(f"  [{h['chunk_id']}] {h['evidence_tier']} | {h['title']} | score {h['score']}")

query: breast ultrasound mass assessment category shape 2D sphericity elongation perimeter gray level co-occurrence texture contrast first order intensity entropy malignancy suspicious benign
  [birads-card:p2:2] primary_reference | ACR BI-RADS Atlas Fifth Edition — Quick Reference | score 1.0
  [birads-card:p2:3] primary_reference | ACR BI-RADS Atlas Fifth Edition — Quick Reference | score 0.7153
  [project-us-lexicon-notes:us-shape-irregular-mass-shape-irregular:0] secondary_project_notes | Project notes — BI-RADS US lexicon paraphrase (Stage 6) | score 0.7862
  [project-us-lexicon-notes:us-shape-round-mass-shape-round:0] secondary_project_notes | Project notes — BI-RADS US lexicon paraphrase (Stage 6) | score 0.583
  [project-us-lexicon-notes:us-probably-benign-solid-probably-benign:0] secondary_project_notes | Project notes — BI-RADS US lexicon paraphrase (Stage 6) | score 0.5511


## 3. Data minimisation: what the LLM is actually sent

In [6]:
facts = rp.build_facts(records[0], records[0]["_prediction"])
payload = json.dumps(rp.build_messages(facts, hits))
from dataset.splits import load_split
raw_ids = [r["image_id"] for s in ("train", "val", "test") for r in load_split(config["paths"]["splits_dir"], s)]
leaks = [x for x in raw_ids + [str(REPO_ROOT), "/Users/", "BUSI class folder", "reference_labels"] if x in payload]
print(f"payload: {len(payload)} chars | raw dataset ids, file paths or labels present: {leaks or 'none'}")
print("top-level fact keys:", list(facts))

payload: 8708 chars | raw dataset ids, file paths or labels present: none
top-level fact keys: ['record_id', 'mask_source', 'mask_quality', 'observed_features', 'quantitative_radiomics_summary', 'classifier_output', 'birads_assessment', 'known_limitations']


## 4. Offline deterministic provider (`stub`): a schema and validator smoke test

In [7]:
def run_report(rec, llm, tag):
    pred = rec["_prediction"]
    hits = retrieve(rec, pred)
    t0 = time.time()
    report = rp.generate_report({k: v for k, v in rec.items() if k != "_prediction"}, pred, hits, llm,
                                max_retries=RESEARCH["reports"]["max_retries"], index_version=index.version)
    report["timing_s"] = round(time.time() - t0, 1)
    stem = f"{rec['record_id']}_{tag}"
    atomic_write_json(out_dir / f"{stem}.json", report)
    (out_dir / f"{stem}.md").write_text(rp.report_to_markdown(report))
    return report

stub_report = run_report(records[0], rp.StubLLM(), "stub")
print("status:", stub_report["status"], "| attempts:", stub_report["attempts"])
assert all(k in stub_report for k in rp.REPORT_SCHEMA_KEYS) and stub_report["requires_radiologist_review"]
print(rp.report_to_markdown(stub_report)[:2500])

status: ok | attempts: 1
# Quantitative ultrasound research report — rec_859f5f43dfe2

> Research model output — not a clinical diagnosis.

- **Status:** ok
- **Requires radiologist review:** yes
- **BI-RADS:** BI-RADS assessment unavailable: validated BI-RADS labels/model not configured.
- **Classifier (benign_vs_malignant, radiomics-gradient-boosting-c622ae6f):** p(malignant) = 0.396 · calibration: calibrated
- **Provider:** stub / template-narrative-v1 · RAG index rag-c47fd3b06bdb · generated 2026-10-29T11:17:50.834262+00:00

## Summary

Quantitative analysis of the lesion outlined by the predicted mask (mask quality: pass). The outlined area is 56114.0 px^2 (pixel units; no physical calibration). The radiomics benign-vs-malignant research model gives p(malignant) = 0.396 (calibration status: calibrated). BI-RADS descriptors were not assessed and no BI-RADS assessment is available.

## BI-RADS descriptors

| Descriptor | Status | Value | Source |
|---|---|---|---|
| shape | Descript

## 5. Local LLM (Qwen2.5-3B-Instruct, cached)

This uses `reporting.llm.LocalLLM` with the model id in `config.yaml → reporting.llm.model_id`, so
nothing leaves the machine. Loading takes about 6 GB of memory, and generation on MPS/CPU takes
minutes. Set `RUN_LOCAL_LLM = False` to skip.
Qwen2.5-3B-Instruct is under the Qwen research licence (non-commercial).

In [8]:
RUN_LOCAL_LLM = True

local_reports = []
if RUN_LOCAL_LLM:
    cfg_local = copy.deepcopy(config)
    cfg_local["research"]["reports"]["llm_backend"] = "local"
    try:
        t0 = time.time()
        llm = rp.build_provider(cfg_local)
        print(f"loaded {llm.model_id} on {llm.device} in {time.time() - t0:.0f}s")
    except Exception as exc:  # noqa: BLE001 - missing weights/deps/memory: report, don't crash
        llm = None
        print("local LLM unavailable:", type(exc).__name__, exc)
    if llm is not None:
        for rec in records:
            r = run_report(rec, llm, "local")
            local_reports.append(r)
            print(f"\n=== {r['record_id']}: status {r['status']} after {r['attempts']} attempt(s), {r['timing_s']}s")
            print(r["summary"] if r["status"] == "ok" else f"error: {r['error']}")
            for e in r["supporting_evidence"]:
                print("  -", e["claim"], e["citations"])

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-3B-Instruct on mps in 10s



=== rec_859f5f43dfe2: status invalid after 3 attempt(s), 404.2s
error: narrative failed validation after 3 attempt(s): schema violation: supporting_evidence.0.claim: String should have at most 400 characters



=== rec_4eb14bcaed4f: status invalid after 3 attempt(s), 221.1s
error: narrative failed validation after 3 attempt(s): supporting_evidence[0] restates a computed value; evidence claims must come from the reference text (put measurements in summary)


In [9]:
if local_reports:
    print(json.dumps({k: local_reports[0][k] for k in ("status", "summary", "supporting_evidence",
                                                      "recommended_review", "birads_assessment",
                                                      "classifier_output")}, indent=1))

{
 "status": "invalid",
 "summary": null,
 "supporting_evidence": [],
 "recommended_review": null,
 "birads_assessment": {
  "status": "unavailable",
  "category": null,
  "source": null,
  "reason": "BI-RADS assessment unavailable: validated BI-RADS labels/model not configured."
 },
 "classifier_output": {
  "available": true,
  "model_id": "radiomics-gradient-boosting-c622ae6f",
  "prediction_type": "benign_vs_malignant",
  "p_malignant": 0.396,
  "calibration_status": "calibrated",
  "out_of_distribution": false,
  "label": "Research model output \u2014 not a clinical diagnosis."
 }
}


## 6. Abstention and validator behaviour

The examples below are **synthetic** and exist only to exercise the safety paths.

In [10]:
import numpy as np

from bus_research import birads
from bus_research.feature_store import build_record
from bus_research.radiomics_extract import RadiomicsSettings, extract_radiomics_features

blank = np.zeros((200, 200, 3), np.uint8)
failed = build_record(image_key="synthetic-empty", lesion_key="0", image=blank,
                      radiomics_result=extract_radiomics_features(blank, np.zeros((200, 200), np.uint8), {},
                                                                  RadiomicsSettings.from_config(config)),
                      segmentation={"model": "unet", "checkpoint_version": None, "mask_source": "predicted",
                                    "threshold": 0.5, "bbox_source": "yolo"},
                      descriptors=birads.empty_descriptors())
abstained = rp.generate_report(failed, {"available": False, "reason": "no lesion"}, [], rp.StubLLM())
print("empty mask ->", abstained["status"], "|", abstained["summary"])

facts = rp.build_facts(records[0], records[0]["_prediction"])
allowed = {h["chunk_id"] for h in hits}
base = json.loads(rp.StubLLM().generate(rp.build_messages(facts, hits)))
bad_cases = {
    "BI-RADS claim": {**base, "summary": base["summary"] + " Overall this is BI-RADS 4B."},
    "management directive": {**base, "recommended_review": "Core biopsy is indicated for this lesion."},
    "invented number": {**base, "summary": base["summary"] + " The mass measures 14.2 mm."},
    "unobserved descriptor": {**base, "summary": base["summary"] + " The margin is spiculated."},
    "unknown citation": {**base, "supporting_evidence": [{"claim": "Some reference claim here.",
                                                          "citations": ["acr-atlas:p99:0"]}]},
}
for name, narrative in bad_cases.items():
    try:
        rp.validate_narrative(json.dumps(narrative), facts, allowed)
        print(f"{name:22s} -> ACCEPTED (unexpected)")
    except rp.NarrativeValidationError as exc:
        print(f"{name:22s} -> rejected: {str(exc)[:110]}")

empty mask -> abstained | No report generated: Radiomics extraction failed (mask contains no lesion pixels).
BI-RADS claim          -> rejected: states a BI-RADS category ('BI-RADS 4') but none is available
management directive   -> rejected: contains a management directive ('biopsy is indicated')
invented number        -> rejected: number 14.2 is not in the facts
unobserved descriptor  -> rejected: summary describes 'spiculated', which was not observed
unknown citation       -> rejected: supporting_evidence[0] cites unknown ids ['acr-atlas:p99:0']


## Limitations

* The validator catches *categories* of unsupported statements (categories, directives, numbers,
  lexicon terms, citations). It cannot verify every paraphrase, and an accepted narrative still needs
  radiologist review (`requires_radiologist_review` is always `true`).
* A 3B local model sometimes fails validation (`status: "invalid"`). In that case nothing is shown,
  and the deterministic facts remain available.
* Evidence is limited to the ACR quick-reference card, PyRadiomics documentation and project notes.
  The full BI-RADS Atlas and peer-reviewed literature are missing evidence (notebook 03).
* Hosted providers (`llm_backend: hf_api`) send the de-identified payload off-machine and require
  `allow_external_llm: true`.